In [1]:
"""
Drop-one-feature importance for REGRESSION (Number of Admissions).

"""
import os
import numpy as np
import pandas as pd
import optuna
import xgboost as xgb
from sklearn.dummy import DummyRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import ElasticNetCV, RidgeCV
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
from sklearn.svm import SVR
from sklearn.model_selection import TimeSeriesSplit
from sklearn.preprocessing import RobustScaler
import warnings

warnings.filterwarnings('ignore')
optuna.logging.set_verbosity(optuna.logging.WARNING)

# --- CONFIGURATION ---
WINSORIZE_QUANTILE = 0.99
LAG_DAYS = [1, 3, 7]
ROLL_WINDOWS = [3, 7]
TARGET_COL = "Number of Admissions"
WIND_DIR_COL = "wind_direction_10m_dominant (°)"
N_SPLITS = 5
N_TRIALS = 25
RANDOM_STATE = 42

np.random.seed(RANDOM_STATE)


# --- 1. DATA LOADING ---
def load_merged_data():
    base_df = pd.read_csv("/Users/suhaniagarwal/Downloads/all_features_data_changed_2.csv")
    extra_weather = pd.read_csv(
        "/Users/suhaniagarwal/Downloads/extra_weather_variables (csv).csv",
        skiprows=3,
        header=0,
    )
    humidity_cols = [
        "time",
        "relative_humidity_2m_mean (%)",
        "relative_humidity_2m_min (%)",
        "relative_humidity_2m_max (%)",
    ]
    humidity_cols = [c for c in humidity_cols if c in extra_weather.columns]
    extra_humidity = extra_weather[humidity_cols].copy()
    base_df["Timestamp"] = pd.to_datetime(base_df["Timestamp"])
    extra_humidity["time"] = pd.to_datetime(extra_humidity["time"])
    merged = base_df.merge(
        extra_humidity,
        left_on="Timestamp",
        right_on="time",
        how="left",
    )
    merged = merged.drop(columns=["time"])
    if "NOx (ppb)" in merged.columns:
        merged = merged.drop(columns=["NOx (ppb)"])
    nh3_cols = [c for c in merged.columns if "NH3" in c]
    if nh3_cols:
        merged = merged.drop(columns=nh3_cols)
    return merged


# --- 2. FEATURE ENGINEERING ---
def engineer_features(df):
    df = df.copy()
    if "NOx (ppb)" in df.columns:
        df = df.drop(columns=["NOx (ppb)"])
    df["Timestamp"] = pd.to_datetime(df["Timestamp"])
    df = df.sort_values("Timestamp").reset_index(drop=True)

    df["dow_sin"] = np.sin(2 * np.pi * df["Timestamp"].dt.dayofweek / 7)
    df["dow_cos"] = np.cos(2 * np.pi * df["Timestamp"].dt.dayofweek / 7)
    df["month_sin"] = np.sin(2 * np.pi * df["Timestamp"].dt.month / 12)
    df["month_cos"] = np.cos(2 * np.pi * df["Timestamp"].dt.month / 12)
    df["month_regime"] = df["Timestamp"].dt.month
    years = df["Timestamp"].dt.year
    year_order = {y: i + 1 for i, y in enumerate(sorted(years.unique()))}
    df["year_regime"] = years.map(year_order)
    df["weekend"] = (df["Timestamp"].dt.dayofweek >= 5).astype(int)

    if WIND_DIR_COL in df.columns:
        df["wind_sin"] = np.sin(2 * np.pi * df[WIND_DIR_COL] / 360)
        df["wind_cos"] = np.cos(2 * np.pi * df[WIND_DIR_COL] / 360)

    exclude = ["Timestamp", TARGET_COL, WIND_DIR_COL, "Date"]
    weather_cols = [
        c
        for c in df.columns
        if c not in exclude and pd.api.types.is_numeric_dtype(df[c])
    ]

    new_feats = {}
    for col in weather_cols:
        for lag in LAG_DAYS:
            new_feats[f"{col}_lag{lag}"] = df[col].shift(lag)
        for roll in ROLL_WINDOWS:
            new_feats[f"{col}_roll{roll}"] = df[col].shift(1).rolling(roll).mean()
    new_feats["admission_lag7"] = df[TARGET_COL].shift(7)

    df = pd.concat([df, pd.DataFrame(new_feats)], axis=1)
    all_nan_cols = [c for c in df.columns if df[c].isna().all()]
    if all_nan_cols:
        df = df.drop(columns=all_nan_cols)
    return df.dropna().reset_index(drop=True)


# --- 3. TUNING ---
def get_optuna_params(model_type, X_tr, y_tr, X_va, y_va):
    y_tr = np.array(y_tr).flatten()
    y_va = np.array(y_va).flatten()

    # region agent log
    try:
        import json, time
        with open("/Users/suhaniagarwal/Downloads/Code_research/.cursor/debug-343183.log", "a") as _f:
            _f.write(json.dumps({
                "sessionId": "343183",
                "runId": "pre-fix",
                "hypothesisId": "H1",
                "location": "Feature_Importance_Regression_script.py:get_optuna_params",
                "message": "enter get_optuna_params",
                "data": {"model_type": str(model_type)},
                "timestamp": int(time.time() * 1000),
            }) + "\n")
    except Exception:
        pass
    # endregion

    def objective(trial):
        if model_type == "rf":
            p = {
                "n_estimators": trial.suggest_int("n_estimators", 50, 200),
                "max_depth": trial.suggest_int("max_depth", 5, 20),
            }
            model = RandomForestRegressor(**p, random_state=RANDOM_STATE, n_jobs=-1)
        elif model_type == "xgb":
            p = {
                "n_estimators": trial.suggest_int("n_estimators", 50, 300),
                "max_depth": trial.suggest_int("max_depth", 3, 10),
                "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.2, log=True),
            }
            model = xgb.XGBRegressor(**p, random_state=RANDOM_STATE)
        model.fit(X_tr, y_tr)
        return r2_score(y_va, model.predict(X_va).flatten())

    study = optuna.create_study(
        direction="maximize",
        sampler=optuna.samplers.TPESampler(seed=RANDOM_STATE, n_startup_trials=10),
    )
    study.optimize(objective, n_trials=N_TRIALS)
    return study.best_params


# --- 4. EXPERIMENT RUNNER ---
def run_experiment(df, experiment_name, drop_cols=None):
    """Run regression CV for one experiment (baseline or drop one base feature). Returns one-row DataFrame."""
    df_local = df.copy()
    if drop_cols:
        existing = [c for c in drop_cols if c in df_local.columns]
        if existing:
            df_local = df_local.drop(columns=existing)

    df_fe = engineer_features(df_local)
    feat_cols = [
        c
        for c in df_fe.columns
        if any(s in c for s in ["_lag", "_roll", "sin", "cos", "weekend", "regime"])
    ]
    X, y = df_fe[feat_cols], df_fe[TARGET_COL]

    tscv = TimeSeriesSplit(n_splits=N_SPLITS)
    fold_results = []
    best_params = {}

    for fold, (train_idx, test_idx) in enumerate(tscv.split(X)):
        X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
        y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

        # Median imputation (train median to train/test) for robustness
        filler = X_train.median()
        X_train = X_train.fillna(filler)
        X_test = X_test.fillna(filler)

        y_train_capped = y_train.clip(upper=y_train.quantile(WINSORIZE_QUANTILE))

        scaler = RobustScaler()
        X_train_s = scaler.fit_transform(X_train)
        X_test_s = scaler.transform(X_test)

        if fold == 0:
            # region agent log
            try:
                import json, time
                with open("/Users/suhaniagarwal/Downloads/Code_research/.cursor/debug-343183.log", "a") as _f:
                    _f.write(json.dumps({
                        "sessionId": "343183",
                        "runId": "pre-fix",
                        "hypothesisId": "H1",
                        "location": "Feature_Importance_Regression_script.py:run_experiment",
                        "message": "tuning models fold0",
                        "data": {"experiment": experiment_name},
                        "timestamp": int(time.time() * 1000),
                    }) + "\n")
            except Exception:
                pass
            # endregion

            print(f"[{experiment_name}] Tuning regression models...")
            split = int(len(X_train_s) * 0.8)
            # Only tune RF and XGB here; SVR uses fixed defaults
            for m in ["rf", "xgb"]:
                best_params[m] = get_optuna_params(
                    m,
                    X_train_s[:split],
                    y_train_capped.iloc[:split],
                    X_train_s[split:],
                    y_train_capped.iloc[split:],
                )

        dummy = DummyRegressor(strategy="mean")
        dummy.fit(X_train_s, y_train_capped)
        preds_d = dummy.predict(X_test_s).flatten()
        scores = {
            "Dummy_R2": r2_score(y_test, preds_d),
            "Dummy_MAE": mean_absolute_error(y_test, preds_d),
            "Dummy_RMSE": np.sqrt(mean_squared_error(y_test, preds_d)),
        }

        models = {
            "Ridge": RidgeCV(alphas=np.logspace(-3, 2, 100), cv=5),
            "ElasticNet": ElasticNetCV(
                cv=5,
                alphas=np.logspace(-4, 0, 50),
                l1_ratio=[0.1, 0.5, 0.7, 0.9, 0.95, 0.99, 1.0],
            ),
            "RandomForest": RandomForestRegressor(**best_params["rf"], random_state=RANDOM_STATE, n_jobs=-1),
            "XGBoost": xgb.XGBRegressor(**best_params["xgb"], random_state=RANDOM_STATE),
            "SVR": SVR(C=1.0, epsilon=0.1),
        }
        for name, model in models.items():
            model.fit(X_train_s, y_train_capped)
            preds = model.predict(X_test_s).flatten()
            scores[f"{name}_R2"] = r2_score(y_test, preds)
            scores[f"{name}_MAE"] = mean_absolute_error(y_test, preds)
            scores[f"{name}_RMSE"] = np.sqrt(mean_squared_error(y_test, preds))

        fold_results.append(scores)
        print(f"[{experiment_name}] Fold {fold+1} complete.")

    metrics_mean = pd.DataFrame(fold_results).mean().to_frame().T
    metrics_mean.insert(0, "experiment", experiment_name)
    return metrics_mean


# --- 5. EXPERIMENT DEFINITIONS ---
def get_base_feature_columns(df):
    exclude = ["Timestamp", TARGET_COL, WIND_DIR_COL, "Date"]
    return [
        c
        for c in df.columns
        if c not in exclude and pd.api.types.is_numeric_dtype(df[c])
    ]


def build_experiments_per_feature(merged_df):
    base_cols = get_base_feature_columns(merged_df)
    experiments = {"baseline": []}
    for col in base_cols:
        experiments[f"drop_{col}"] = [col]
    return experiments


def recommend_drops(comparison, metric="XGBoost_R2", top_n_drop=10):
    """Recommend features to drop: smallest drop in R2 (or gain) when removed = safe to drop."""
    if "experiment" not in comparison.columns or metric not in comparison.columns:
        return
    baseline_row = comparison.loc[comparison["experiment"] == "baseline"]
    if baseline_row.empty:
        return
    baseline_val = float(baseline_row[metric].iloc[0])
    drop_rows = comparison.loc[comparison["experiment"] != "baseline"].copy()
    drop_rows["_drop_in_metric"] = baseline_val - drop_rows[metric]
    drop_rows = drop_rows.sort_values("_drop_in_metric", ascending=True)
    low_importance = drop_rows.head(top_n_drop)
    high_importance = drop_rows.tail(min(top_n_drop, len(drop_rows))).iloc[::-1]

    print(f"\n=== Feature drop recommendation (by {metric} vs baseline) ===")
    print("Consider DROPPING (lowest importance; removing these hurts least or helps):")
    for _, r in low_importance.iterrows():
        name = r["experiment"].replace("drop_", "")
        print(f"  {name}  (Δ = {r['_drop_in_metric']:+.4f})")
    print("\nKeep (highest importance; removing these hurts most):")
    for _, r in high_importance.iterrows():
        name = r["experiment"].replace("drop_", "")
        print(f"  {name}  (Δ = {r['_drop_in_metric']:+.4f})")


def run_all_experiments():
    merged = load_merged_data()
    experiments = build_experiments_per_feature(merged)
    print(f"Running {len(experiments)} experiments (1 baseline + {len(experiments)-1} single-feature drops).\n")
    results = []

    for name, drop_cols in experiments.items():
        print(f"\n=== Running experiment: {name} ===")
        metrics = run_experiment(merged, name, drop_cols)
        results.append(metrics)

    comparison = pd.concat(results, ignore_index=True)
    print("\n=== Experiment comparison (mean CV metrics) ===")
    print(comparison)
    recommend_drops(comparison, metric="XGBoost_R2", top_n_drop=10)
    return comparison


if __name__ == "__main__":
    run_all_experiments()

/opt/miniconda3/envs/research2/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Running 22 experiments (1 baseline + 21 single-feature drops).


=== Running experiment: baseline ===
[baseline] Tuning regression models...
[baseline] Fold 1 complete.
[baseline] Fold 2 complete.
[baseline] Fold 3 complete.
[baseline] Fold 4 complete.
[baseline] Fold 5 complete.

=== Running experiment: drop_PM2.5 (µg/m³) ===
[drop_PM2.5 (µg/m³)] Tuning regression models...
[drop_PM2.5 (µg/m³)] Fold 1 complete.
[drop_PM2.5 (µg/m³)] Fold 2 complete.
[drop_PM2.5 (µg/m³)] Fold 3 complete.
[drop_PM2.5 (µg/m³)] Fold 4 complete.
[drop_PM2.5 (µg/m³)] Fold 5 complete.

=== Running experiment: drop_PM10 (µg/m³) ===
[drop_PM10 (µg/m³)] Tuning regression models...
[drop_PM10 (µg/m³)] Fold 1 complete.
[drop_PM10 (µg/m³)] Fold 2 complete.
[drop_PM10 (µg/m³)] Fold 3 complete.
[drop_PM10 (µg/m³)] Fold 4 complete.
[drop_PM10 (µg/m³)] Fold 5 complete.

=== Running experiment: drop_NO (µg/m³) ===
[drop_NO (µg/m³)] Tuning regression models...
[drop_NO (µg/m³)] Fold 1 complete.
[drop_NO (µg/m³)] Fold 2 c